# California State Procurement Data — Preparation for AI Agent

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pymongo import MongoClient, ASCENDING

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

df = pd.read_csv(
    r"C:\Users\SP7\Downloads\archive (1)\PURCHASE ORDER DATA EXTRACT 2012-2015_0.csv",
    low_memory=False
)
print(f"Loaded {len(df):,} rows x {len(df.columns)} columns")

## 1. Basic Info

In [ ]:
print(df.shape)

We have **346,018 purchase orders** and **31 columns** describing each one.

In [ ]:
print(df.dtypes)

Most columns are stored as text (`object`). The ones that need fixing:
- **Total Price** and **Unit Price** — text because of `$`. Must be numbers for aggregation queries.
- **Creation Date** and **Purchase Date** — text. Must stay as native datetime objects for MongoDB date queries.

In [ ]:
df.head(5)

A quick look at the first 5 rows. Dates like `08/27/2013` and prices like `$1.00` are plain text and need cleaning before loading into MongoDB.

## 2. Data Quality

In [ ]:
df.isnull().sum()

Key missing data:
- **Purchase Date** — 17,436 missing (~5%). Many orders have no delivery date.
- **Requisition Number** — 331,649 missing (~96%). Not reliable to query on.
- **LPA Number** — 253,673 missing (~73%). Same issue.
- **Item Name, Total Price, Unit Price** — fewer than 35 missing each. Very clean.

In [ ]:
df.duplicated().sum()

There are **2,084 duplicate rows**. We'll remove them in cleaning to avoid double-counting in aggregations.

## 3. Cleaning

In [ ]:
# Remove $ and commas, convert prices to float
df['Total Price'] = df['Total Price'].str.replace(r'[\$,]', '', regex=True).astype(float)
df['Unit Price']  = df['Unit Price'].str.replace(r'[\$,]', '', regex=True).astype(float)

print(df[['Unit Price', 'Total Price']].head(5))

Prices are now real numbers. We can now calculate totals, averages, and run range queries.

In [ ]:
# Convert dates to datetime — kept as datetime objects for native MongoDB date support
df['Creation Date'] = pd.to_datetime(df['Creation Date'], errors='coerce')
df['Purchase Date'] = pd.to_datetime(df['Purchase Date'], errors='coerce')

print(df[['Creation Date', 'Purchase Date']].dtypes)
print(df[['Creation Date', 'Purchase Date']].head(5))

Dates are now proper datetime objects — **not strings**. MongoDB stores these as native ISODate, enabling filtering by month, year, quarter, or any date range.

In [ ]:
# Normalize Item Name — lowercase and strip whitespace
df['Item Name'] = df['Item Name'].str.lower().str.strip()

# Strip whitespace from all string columns
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

# Drop duplicates
df = df.drop_duplicates()
print(f"Rows after removing duplicates: {len(df):,}")

Item names are now all lowercase — "Toner" and "toner" count as one. Duplicates removed. Dataset is clean.

In [ ]:
# Rename all columns to snake_case for machine-friendly MongoDB field names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r'[\s\-/]+', '_', regex=True)
    .str.replace(r'[^a-z0-9_]', '', regex=True)
)
print(list(df.columns))

All column names are now `snake_case`. The AI agent can reference field names directly in MongoDB queries without escaping.

In [ ]:
# Add helper columns — year, month, quarter as separate integers (not combined strings)
# This makes MongoDB grouping and filtering much simpler:
#   Filter by year:    { year: 2014 }
#   Group by quarter:  { $group: { _id: { year: '$year', quarter: '$quarter' } } }
df['year']    = df['creation_date'].dt.year.astype('Int64')
df['month']   = df['creation_date'].dt.month.astype('Int64')
df['quarter'] = df['creation_date'].dt.quarter.astype('Int64')

print(df[['creation_date', 'year', 'month', 'quarter']].head(5))

Year, month, and quarter are stored as separate integers. This avoids string parsing in MongoDB and makes time-based aggregations fast and straightforward.

## 4. EDA — Exploratory Data Analysis

This section explores the dataset from the angle of the queries the AI agent will need to answer:
spending trends, top suppliers, most frequent items, time patterns, and category breakdowns.

### 4.1 Distribution of Total Price
*Supports queries like: "What is the typical order size?" or "Show me orders above $50,000."*

In [ ]:
print(df['total_price'].describe())
print(f"\nSkewness : {df['total_price'].skew():.2f}")
print(f"Median   : ${df['total_price'].median():,.2f}")
print(f"Mean     : ${df['total_price'].mean():,.2f}")

The distribution is **heavily right-skewed** — the mean is much higher than the median, meaning a small number of very large orders are pulling the average up.
Most orders are small, but a few are extremely large. This is important context for the AI agent when answering questions about "average spending."

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Log-scale histogram — needed because of the extreme skew
clean_prices = df['total_price'].dropna()
clean_prices[clean_prices > 0].apply(np.log10).hist(bins=50, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Total Price (log10 scale)')
axes[0].set_xlabel('log10(total_price)')
axes[0].set_ylabel('Number of orders')

# Box plot to show spread and outliers
df.boxplot(column='total_price', ax=axes[1], vert=True)
axes[1].set_title('Total Price — Box Plot')
axes[1].set_ylabel('total_price ($)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

The histogram (log scale) shows most orders fall between $10 and $10,000. The box plot makes the extreme outliers visible — those are the capped `$999,999` entries noted earlier.

In [ ]:
# IQR-based outlier detection
Q1  = df['total_price'].quantile(0.25)
Q3  = df['total_price'].quantile(0.75)
IQR = Q3 - Q1
upper_fence = Q3 + 1.5 * IQR

outliers = df[df['total_price'] > upper_fence]
print(f"IQR upper fence : ${upper_fence:,.2f}")
print(f"Outlier rows    : {len(outliers):,}  ({len(outliers)/len(df)*100:.1f}% of dataset)")
print(f"Outlier spend   : ${outliers['total_price'].sum():,.2f}")
print(f"\nSuspected capped values (total_price >= $999,000): {(df['total_price'] >= 999_000).sum():,}")

**Outlier summary:** Using the IQR method, any order above the upper fence is considered an outlier.
These records are kept but flagged. The AI agent should note this when answering questions about total spending — a small number of likely-capped entries inflate the total significantly.

### 4.2 Time-Based Analysis
*Supports queries like: "How many orders were placed in Q3 2014?" or "Which month had the highest spending?"*

In [ ]:
# Orders per year
orders_per_year = df.groupby('year').size()

orders_per_year.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Number of Orders per Year')
plt.xlabel('Year')
plt.ylabel('Number of orders')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(orders_per_year)

Order volume is **relatively stable across the 3 years**, with a slight increase in 2014. The `year` field in MongoDB can be queried directly: `{ year: 2014 }`.

In [ ]:
# Orders per month (across all years)
orders_per_month = df.groupby('month').size()

orders_per_month.plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Number of Orders by Month (all years combined)')
plt.xlabel('Month')
plt.ylabel('Number of orders')
plt.xticks(ticks=range(12), labels=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], rotation=0)
plt.tight_layout()
plt.show()

print(orders_per_month)

**June is the busiest month by far** — the end of California's fiscal year, when departments rush to spend remaining budgets. This seasonal spike is important context for the AI agent answering time-based queries.

In [ ]:
# Monthly spending trend over time
monthly_spend = df.groupby(df['creation_date'].dt.to_period('M'))['total_price'].sum()
monthly_spend.index = monthly_spend.index.to_timestamp()

monthly_spend.plot(kind='line', color='steelblue', linewidth=1.5)
plt.title('Monthly Spending Trend')
plt.xlabel('Date')
plt.ylabel('Total Spend ($)')
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
plt.tight_layout()
plt.show()

Spending spikes every June — consistent with the fiscal year-end pattern. The AI agent should flag this seasonality when answering questions like "which period had the highest spending."

In [ ]:
# Spending by quarter (year + quarter)
quarterly_spend = df.groupby(['year', 'quarter'])['total_price'].sum().reset_index()
quarterly_spend['label'] = quarterly_spend['year'].astype(str) + '-Q' + quarterly_spend['quarter'].astype(str)

quarterly_spend.set_index('label')['total_price'].plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Total Spending by Quarter')
plt.xlabel('Quarter')
plt.ylabel('Total Spend ($)')
plt.gca().yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.0f}M'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(quarterly_spend[['label','total_price']].sort_values('total_price', ascending=False).head(5).to_string(index=False))

Q4 (April–June) is consistently the highest-spending quarter across all years. In MongoDB, this maps to: `{ $match: { quarter: 4 } }`.

### 4.3 Categorical Analysis
*Supports queries like: "Who are the top suppliers?" or "What items are ordered most?"*

In [ ]:
# Top 10 suppliers by number of orders
top_suppliers = df['supplier_name'].value_counts().head(10)

top_suppliers.sort_values().plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Top 10 Suppliers by Order Count')
plt.xlabel('Number of orders')
plt.tight_layout()
plt.show()

print(top_suppliers)

These are the most frequently appearing suppliers in the dataset. Frequency of orders doesn't necessarily mean highest spend — a supplier could place many small orders.

In [ ]:
# Top 20 most ordered items
top_items = df['item_name'].value_counts().head(20)

top_items.sort_values().plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Top 20 Most Ordered Items')
plt.xlabel('Number of orders')
plt.tight_layout()
plt.show()

print(top_items)

Medical supplies, toner, and office supplies dominate. After lowercasing, variants like "Toner"/"toner"/"toners" are separate — a `$regex` query in MongoDB can group them if needed.

In [ ]:
# Top acquisition types (categories)
print("--- Acquisition Type ---")
print(df['acquisition_type'].value_counts())
print("\n--- Acquisition Method ---")
print(df['acquisition_method'].value_counts().head(10))

**Acquisition Type** (IT Goods, NON-IT Goods, IT Services, etc.) and **Acquisition Method** (Statewide Contract, Informal Competitive, etc.) are clean categorical fields — well-suited for MongoDB `$match` filters and `$group` aggregations.

### 4.4 Data Consistency Checks
*Inconsistencies in supplier names would cause the AI agent to return incomplete results — e.g., "AT&T" and "at&t" treated as different suppliers.*

In [ ]:
# Check for case variations in supplier_name
# Group by lowercased name and count how many distinct cased versions exist
supplier_case_issues = (
    df.groupby(df['supplier_name'].str.lower())['supplier_name']
    .nunique()
)
multi_case = supplier_case_issues[supplier_case_issues > 1].sort_values(ascending=False)

print(f"Suppliers with casing inconsistencies: {len(multi_case)}")
print(multi_case.head(10))

Any supplier appearing here has multiple casing variants (e.g., "State Fund" vs "state fund"). We'll normalize all supplier names to title case in the next step so the AI agent gets consistent results.

In [ ]:
# Fix: normalize supplier_name and department_name to title case
df['supplier_name']   = df['supplier_name'].str.title()
df['department_name'] = df['department_name'].str.title()

# Verify — re-run the check
supplier_case_issues = (
    df.groupby(df['supplier_name'].str.lower())['supplier_name']
    .nunique()
)
print(f"Remaining casing inconsistencies: {(supplier_case_issues > 1).sum()}")

Supplier and department names are now consistently title-cased. The AI agent will get the same results whether the user types "state fund" or "State Fund".

In [ ]:
# Check for extra spaces in supplier_name (already stripped but worth confirming)
has_extra_spaces = df['supplier_name'].str.contains(r'  ', regex=False).sum()
print(f"Supplier names with extra internal spaces: {has_extra_spaces}")

Confirms no double-spaces remain in supplier names after our earlier strip pass.

### 4.5 Relationships Between Features
*Supports queries like: "Which supplier has the highest total spend?" or "Which category costs the most?"*

In [ ]:
# Spending by supplier — top 10
spend_by_supplier = df.groupby('supplier_name')['total_price'].sum().sort_values(ascending=False).head(10)

spend_by_supplier.sort_values().plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Top 10 Suppliers by Total Spend')
plt.xlabel('Total Spend ($)')
plt.gca().xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
plt.tight_layout()
plt.show()

print(spend_by_supplier.apply(lambda x: f'${x:,.2f}'))

This is the most direct answer to "top suppliers by spend" — a common procurement query. In MongoDB this maps to a `$group` + `$sort` aggregation on `total_price`.

In [ ]:
# Spending by acquisition type (category)
spend_by_type = df.groupby('acquisition_type')['total_price'].sum().sort_values(ascending=False)

spend_by_type.sort_values().plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Total Spend by Acquisition Type')
plt.xlabel('Total Spend ($)')
plt.gca().xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.0f}M'))
plt.tight_layout()
plt.show()

print(spend_by_type.apply(lambda x: f'${x:,.2f}'))

NON-IT Goods and IT Services dominate total spend. The AI agent can use `acquisition_type` as a reliable filter field for category-based queries.

In [ ]:
# Spending by department — top 10
spend_by_dept = df.groupby('department_name')['total_price'].sum().sort_values(ascending=False).head(10)

spend_by_dept.sort_values().plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Top 10 Departments by Total Spend')
plt.xlabel('Total Spend ($)')
plt.gca().xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.0f}M'))
plt.tight_layout()
plt.show()

print(spend_by_dept.apply(lambda x: f'${x:,.2f}'))

Health & Human Services and Transportation are the top-spending departments — expected given the scale of their operations. `department_name` is a key filter field for the AI agent.

## Load to MongoDB

In [ ]:
client     = MongoClient('mongodb://localhost:27017/')
collection = client['procurement_db']['orders']

# delete_many instead of drop() — preserves collection structure and indexes
collection.delete_many({})

records = df.to_dict(orient='records')
result  = collection.insert_many(records)
print(f"Inserted {len(result.inserted_ids):,} records into procurement_db.orders")

Records are inserted. Using `delete_many({})` preserves collection structure and any existing indexes between reloads.

In [ ]:
# Indexes on the fields the AI agent will query most frequently
collection.create_index([('creation_date',  ASCENDING)], name='idx_creation_date')
collection.create_index([('supplier_name',  ASCENDING)], name='idx_supplier_name')
collection.create_index([('total_price',    ASCENDING)], name='idx_total_price')
collection.create_index([('department_name',ASCENDING)], name='idx_department_name')
collection.create_index([('year',           ASCENDING)], name='idx_year')
collection.create_index([('quarter',        ASCENDING)], name='idx_quarter')

print("Indexes created:")
for name in collection.index_information():
    print(f"  - {name}")

client.close()

Six indexes added — covering the fields the AI agent will most commonly filter, sort, and group by:
- `creation_date`, `year`, `quarter` — all time-based queries
- `supplier_name`, `department_name` — categorical lookups
- `total_price` — range queries and spend aggregations

The MongoDB collection is now clean, structured, and optimized for the AI procurement agent.